# Bright Coffee Shop - Control Flow Practical
**BrightLearn · Python for Data Analytics**

---

In this practical you will use real coffee shop sales data to practise Python control flow.  
Every task is set in a real business scenario - the kind of analysis you would actually do on the job.

| Task | Topic | Scenario |
|------|-------|----------|
| Setup | Load data | Get the dataset ready |
| 1 | `if / elif / else` | Flag a single transaction by size |
| 2 | `if / elif / else` + DataFrame | Flag every transaction in the dataset |
| 3 | `for` loop | Print a revenue summary for each store |
| 4 | `for` + `if` inside loop | Find the busiest hours of the day |
| 5 | `while` loop | Find when cumulative revenue crosses a target |
| 6 | Functions | Write a reusable store report |
| * | Bonus | Upgrade the function with more stats |

> **How to work through this notebook:**  
> 1. Read the scenario and concept reminder in each task  
> 2. Try to fill in the blanks yourself  
> 3. Run the cell and check your output matches the expected result  
> 4. Ask your instructor if you are stuck - then we reveal together

---
## Setup - Run This First

Run this cell before anything else. It loads the dataset and creates the columns we will need throughout the practical.

You should see:
- The **shape** of the dataset (rows, columns)
- The **first 3 rows** of data
- A list of **column names and their data types**

In [0]:
import pandas as pd

# Load the dataset
# sep=';' because this CSV uses semicolons instead of commas
# decimal=',' because decimal points are written as commas in this file
df = spark.read.table("dataset.default.bright_coffee_shop_sales").toPandas()

# Parse dates and extract useful columns
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['hour']    = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour
df['month']   = df['transaction_date'].dt.month
df['revenue'] = df['unit_price'] * df['transaction_qty']

# Quick check - always run this after loading
print('Shape:', df.shape)
print()
print(df.head(3))
print()
print(df.dtypes)

Shape: (149116, 14)

   transaction_id transaction_date    transaction_time  ...  hour  month revenue
0               1       2023-01-01 2026-05-27 07:06:11  ...     7      1      33
1               2       2023-01-01 2026-05-27 07:08:56  ...     7      1  3,13,1
2               3       2023-01-01 2026-05-27 07:14:04  ...     7      1  4,54,5

[3 rows x 14 columns]

transaction_id               int64
transaction_date    datetime64[ns]
transaction_time    datetime64[ns]
transaction_qty              int64
store_id                     int64
store_location              object
product_id                   int64
unit_price                  object
product_category            object
product_type                object
product_detail              object
hour                         int64
month                        int64
revenue                     object
dtype: object


---
## Task 1 - `if / elif / else`
### Flag a Single Transaction by Revenue Size

**Scenario:**  
The store manager wants to classify transactions by size:
- Revenue **above R10** -> `'High'`
- Revenue **between R5 and R10** (inclusive of R5) -> `'Medium'`  
- Revenue **below R5** -> `'Low'`

---

**Concept reminder:**

```python
if condition:        # colon is required
    # 4 spaces indent
elif other_condition:
    # only checked if the if above was False
else:
    # catches everything that did not match
```

- `=` stores a value. `==` checks equality. They are **not** the same.
- Python checks conditions **top to bottom** and stops at the first True one.

---

**Your task:**
1. Fill in the three conditions (`___`) using `>` and `>=` with the correct thresholds
2. Fill in the three string labels
3. Run the cell - does the output say `Medium`?
4. Change `revenue` to `3`, `10`, and `12` and re-run each time

In [0]:
revenue = 8.50  # test value — change this to test all three branches

if revenue >= 10: # 8.50 being greater than 10 is FALSE
    flag = "High"
elif revenue >= 5 and revenue <= 10: # 8.50 being greater than 5 TRUE and being less than 10 TRUE
    flag = "Medium"
else:
    flag ="Low" 

print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R8.5 → Flag: Medium


In [0]:
revenue = 3

if revenue >= 10: # 3 being greater than  10 FALSE
    flag = "High"
elif revenue >= 5 and revenue <= 10: # 3 being greater than 5 FALSE and 3 being less than 10 TRUE
    flag = "Medium"
else:
    flag ="Low"

print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R3 → Flag: Low


In [0]:
revenue = 10

if revenue >= 10: # 10 being greater than 10 TRUE
    flag = "High"
elif revenue >= 5 and revenue <= 10: # 10 being greater than 5 TRUE and 10 being less than 10 TRUE
    flag = "Medium"
else:
    flag ="Low"

print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R10 → Flag: High


In [0]:
revenue = 12 
if revenue >= 10: # 12 being greater than 12 TRUE
    flag = "High"
elif revenue >= 5 and revenue <= 10: #  12 being greater than 5 TRUE and 12 bring less than 10 TRUE
    flag = "Medium"
else:
    flag ="Low"

print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R12 → Flag: High


In [0]:
revenue = 12 
if revenue >= 10: # 12 being greater than 12 TRUE
    flag = "High"
elif revenue >= 5 and revenue <= 10: #  12 being greater than 5 TRUE and 12 bring less than 10 TRUE
    flag = "Medium"
else:
    flag ="Low"

print(f'Revenue: R{revenue} → Flag: {flag}')

Revenue: R12 → Flag: High


**Expected output for `revenue = 8.50`:**
```
Revenue: R8.5 -> Flag: Medium
```

**Think about it:**
- What happens if `revenue` is exactly `5`? Exactly `10`?
- What would happen if you swapped the `if` and `elif` blocks?

# Answer
- If revenue is exactly 5 the Revenue would be Low and for 10 the revenue would be High
- You will get a syntax error because elif cannot come before an if.

---
## Task 2 - if / elif / else applied to the DataFrame
### Flag Every Transaction in the Dataset

**Scenario:**
Apply the same logic from Task 1 to every row in the dataset.
Create a new column called `revenue_flag` that classifies every transaction as `'High'`, `'Medium'`, or `'Low'`.

---

**Concept reminder:**
We can write a regular function and use `.apply()` to run it on every row of a column:

```python
def classify(value):
    if value > 10:
        return 'High'
    elif value >= 5:
        return 'Medium'
    else:
        return 'Low'

df['new_col'] = df['col'].apply(classify)
```

- `def` defines the function - nothing runs yet
- `.apply(classify)` calls the function once for every value in the column
- The function receives each value one at a time as its parameter

---

**Your task:**
1. Fill in the three conditions inside `classify_revenue` using the same thresholds as Task 1
2. Fill in the three return labels
3. Apply the function to the revenue column
4. Run the cell and check the value counts

In [0]:
def classify_revenue(value):
    if value > 10:
        return 'High'
    elif value >= 5 and value <= 10:
        return 'Medium'
    else:
        return 'Low'
#  converting the column to a number before applying the function.
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')
df['revenue_flag'] = df['revenue'].apply(classify_revenue)

# How many transactions fall into each category?
print(df['revenue_flag'].value_counts())
print()

# Spot-check: do the labels match the revenue values?
print(df[['revenue', 'revenue_flag']].head(10))

Low       126908
High       22074
Medium       134
Name: revenue_flag, dtype: int64

   revenue revenue_flag
0     33.0         High
1      NaN          Low
2      NaN          Low
3      2.0          Low
4      NaN          Low
5      3.0          Low
6      2.0          Low
7     22.0         High
8      NaN          Low
9      NaN          Low


**Expected output (approximate):**
```
High      XXXX
Medium    XXXX
Low       XXXX
Name: revenue_flag, dtype: int64
```

**Think about it:**
- What does `.apply(classify_revenue)` do differently from calling `classify_revenue(8.50)` directly?
- What would happen if you misspelled a return value like `'high'` instead of `'High'`?

# Answer
- .apply(classify_revenue) runs the function on every value in the column, while classify_revenue(8.50) only checks one value.

- it will treat 'high' and 'High' as different values because it is case sensitive

---
## Task 3 - `for` loop
### Print a Revenue Summary for Each Store

**Scenario:**  
The regional manager wants a quick verbal summary of each store's performance.  
Loop through each unique store location and print its total revenue.

---

**Concept reminder:**

```python
for item in collection:   # colon required, 4-space indent inside
    print(item)           # this runs once for every item
```

- The variable after `for` (here `item`) is your choice - name it clearly
- `df['col'].unique()` returns an array of unique values - great for looping
- `df[df['col'] == value]['other_col'].sum()` filters and sums in one line

---

**Your task:**
1. Fill in the loop variable name (choose something descriptive)
2. Fill in the filter condition inside the loop
3. Fill in the column to sum for revenue
4. Fill in the f-string to print a clear summary line

In [0]:
stores = df['store_location'].unique()
print('Stores found:', stores)
print()

for stores in stores:
    store_data = df[df['store_location'] == stores]
    total      = store_data['revenue'].sum()
    print(f'{stores}: R{total:,.2f}')

Stores found: ['Lower Manhattan' "Hell's Kitchen" 'Astoria']

Lower Manhattan: R363,665,236,549.00
Hell's Kitchen: R45,454,545,454,805,944.00
Astoria: R250,898.00


**Expected output:**
```
Hells Kitchen    -> Total Revenue: RXXXXXX.XX
Astoria          -> Total Revenue: RXXXXXX.XX
Lower Manhattan  -> Total Revenue: RXXXXXX.XX
```
*Tip: use `{total:,.2f}` in your f-string for a nicely formatted number.*

**Think about it:**
- How many times does the loop run? How do you know?
- What is stored in the loop variable on each iteration?

# Answer
- The loop runs once for each item in the collection. I know this because a loop goes through every item one at a time until there are no more items left.

- The loop variable stores the current item being processed in the collection. It changes on each iteration as the loop moves to the next item.


---
## Task 4 - `for` loop with `if` inside
### Find the Busiest Hours of the Day

**Scenario:**  
The operations team wants to know which hours need extra staff.  
Loop through every hour (0-23) and print only the hours where transaction count exceeds **3,000**.

---

**Concept reminder:**  
You can put an `if` statement *inside* a `for` loop:

```python
for i in range(24):          # loops 0, 1, 2 ... 23
    if some_condition:       # checked on every iteration
        print(i)             # only runs when condition is True
```

- `range(24)` gives integers 0 through 23 - one per hour of the day
- `len(df[df['hour'] == h])` counts how many rows match that hour

---

**Your task:**
1. Fill in `range()` with the right number for 24 hours
2. Fill in the filter to count transactions for hour `h`
3. Fill in the threshold condition
4. Print a clear line showing the hour and count

In [0]:
print('Peak hours (more than 3,000 transactions):')
print()

for h in range(24):
    count = len(df[df['hour'] == h])
    if count > 3000:
        print(f'  Hour {h:02d}:00  →  {count:,} transactions')

Peak hours (more than 3,000 transactions):

  Hour 06:00  →  4,594 transactions
  Hour 07:00  →  13,428 transactions
  Hour 08:00  →  17,654 transactions
  Hour 09:00  →  17,764 transactions
  Hour 10:00  →  18,545 transactions
  Hour 11:00  →  9,766 transactions
  Hour 12:00  →  8,708 transactions
  Hour 13:00  →  8,714 transactions
  Hour 14:00  →  8,933 transactions
  Hour 15:00  →  8,979 transactions
  Hour 16:00  →  9,093 transactions
  Hour 17:00  →  8,745 transactions
  Hour 18:00  →  7,498 transactions
  Hour 19:00  →  6,092 transactions


**Expected output:**
```
Peak hours (more than 3,000 transactions):

  Hour 08:00  ->  X,XXX transactions
  Hour 09:00  ->  X,XXX transactions
  Hour 10:00  ->  X,XXX transactions
  ...
```

**Think about it:**
- How many total iterations does the loop do?
- How many lines does it *print*? Why are those different numbers?
- What would you change to find the *quietest* hours instead?

# Answer
- The loop does one iteration for each item in the collection.
- It only prints lines when the condition is true. The number of iterations can be bigger because the loop checks every item, but it does not print every item.
- I would change the condition so that it looks for the lowest values instead of the highest values.

---
## Task 5 - `while` loop
### Find When Cumulative Revenue Crosses a Target

**Scenario:**  
The business set a cumulative revenue target of **R150,000**.  
Starting from month 1, keep adding monthly revenue until the running total exceeds the target.  
Report which month it crosses the threshold.

---

**Concept reminder:**

```python
while condition:      # runs as long as condition is True
    # do something
    counter += 1      # CRITICAL: always update something inside
                      # or the loop runs forever
```

- Use `while` when you don't know upfront how many iterations you need
- Something inside the loop **must** eventually make the condition False
- `x += 1` is shorthand for `x = x + 1`

---

**Your task:**
1. Fill in the `while` condition - loop while cumulative is below the target
2. Fill in the column name to sum for monthly revenue
3. Fill in what to add to `cumulative` each iteration
4. Fill in the increment so month advances by 1 each time

In [0]:
target     = 150_000
month      = 1
cumulative = 0
max_month  = df['month'].max()   # safety — don't go past the data

while cumulative < target and month <= max_month:
    monthly_rev = df[df['month'] == month]['revenue'].sum()
    cumulative += monthly_rev
    print(f'Month {month}: +R{monthly_rev:,.0f}  →  Running total: R{cumulative:,.0f}')
    month += 1

print()
print(f'✓ Target of R{target:,} crossed at month {month - 1}! Final total: R{cumulative:,.0f}')

Month 1: +R9,090,909,105,177,048  →  Running total: R9,090,909,105,177,048

✓ Target of R150,000 crossed at month 1! Final total: R9,090,909,105,177,048


**Expected output:**
```
Month 1: +RXXXXX  ->  Running total: RXXXXX
Month 2: +RXXXXX  ->  Running total: RXXXXXX
...
v Target of R150,000 crossed at month X! Final total: RXXX,XXX
```

**Think about it:**
- Why is `and month <= max_month` included? What would happen without it?
- What would happen if you deleted `month += 1`? *(Don't try this - it will freeze your cell!)*
- Why use `while` here instead of `for`?

# Answer
- It makes sure the loop stops when it reaches the maximum month. Without it, the loop could keep running past the allowed months.
- The value of month would never change, so the loop condition would always stay true and the loop would run forever.
- A while loop is used because the loop continues as long as a condition is true. A for loop is usually used when you already know how many times you want to repeat something.

---
## Task 6 - Functions
### Write a Reusable Store Report Function

**Scenario:**  
You need to produce a summary report for any store on demand.  
Instead of rewriting the same code three times, write a function called `store_report`  
that takes a store name and prints its key statistics.

---

**Concept reminder:**

```python
def function_name(parameter):   # define the function - nothing runs yet
    result = parameter * 2      # parameter is a placeholder
    print(result)               # no return needed if just printing

function_name(10)               # NOW it runs - 10 flows in as parameter
```

- `def` defines the function. Calling the name **runs** it. These are two separate steps.
- The parameter (e.g. `store_name`) is a placeholder - it holds whatever you pass in
- The code inside only runs **when you call the function**

---

**Your task:**
1. Fill in the parameter name in `def store_report(___)`
2. Use that parameter to filter the DataFrame inside the function
3. Calculate total revenue, transaction count, and average revenue per transaction
4. Print the stats neatly
5. Call the function for all three stores

In [0]:
def store_report(name):
    # Filter to just this store
    data  = df[df['store_location'] == 'name']

    # Calculate stats
    total = data['revenue'].sum()
    count = len(data)
    avg   = total / count

    # Print the report
    print(f'── {name} ──')
    print(f'  Total revenue   : R{total:,.2f}')
    print(f'  Transactions    : {count:,}')
    print(f'  Avg per sale    : R{avg:.2f}')
    print()


# Call the function for each store
store_report('Hells Kitchen')
store_report('Astoria')
store_report('Lower Manhattan')

── Hells Kitchen ──
  Total revenue   : R0.00
  Transactions    : 0
  Avg per sale    : Rnan

── Astoria ──
  Total revenue   : R0.00
  Transactions    : 0
  Avg per sale    : Rnan

── Lower Manhattan ──
  Total revenue   : R0.00
  Transactions    : 0
  Avg per sale    : Rnan



/home/spark-97442477-81ca-4b7c-bf0e-7b/.ipykernel/3074/command-7940166542406233-2898828730:8: RuntimeWarning: invalid value encountered in double_scalars
  avg   = total / count
/home/spark-97442477-81ca-4b7c-bf0e-7b/.ipykernel/3074/command-7940166542406233-2898828730:8: RuntimeWarning: invalid value encountered in double_scalars
  avg   = total / count
/home/spark-97442477-81ca-4b7c-bf0e-7b/.ipykernel/3074/command-7940166542406233-2898828730:8: RuntimeWarning: invalid value encountered in double_scalars
  avg   = total / count


**Expected output:**
```
── Hells Kitchen ──
  Total revenue   : RXXXXXX.XX
  Transactions    : XX,XXX
  Avg per sale    : RX.XX

── Astoria ──
  ...

── Lower Manhattan ──
  ...
```

**Think about it:**
- What happens if you run `def store_report(store_name):` on its own without calling it?
- What would happen if you typed `store_report('Hell Kitchen')` (missing the 's')?
- How is writing a function different from writing the same code three times?

# Answer
- It only creates the function. The code inside the function will not run until the function is called.
- The function would still run, but it would use 'Hell Kitchen' as the store name, which might give incorrect results if the correct name is 'Hells Kitchen'.
- A function lets you write the code once and reuse it whenever you need it. This makes the code shorter, easier to read, and easier to update.

---
## * Bonus - Upgrade the Function
### Add the Top-Selling Product Category to the Report

**Scenario:**  
Extend `store_report` to also show the **top-selling product category** for each store.

---

**Hint:**  
Inside the function, after filtering to `data`, group by `product_category` and sum revenue.  
Then use `.idxmax()` to get the name of the category with the highest total.

```python
top_cat = data.groupby('product_category')['revenue'].sum().idxmax()
```

Add a line to the print output:  
```
  Top category    : Coffee
```

---

Write your upgraded function from scratch in the cell below:

In [0]:
def store_report(name):
    data = df[df['store'] == 'name']

    total_revenue = data['revenue'].sum()
    total_orders = data['order_id'].count()
    avg_order = total_revenue / total_orders

    top_cat = data.groupby('product_category')['revenue'].sum().idxmax()

    print("Store:",name)
    print("Total revenue   :", total_revenue)
    print("Total orders    :", total_orders)
    print("Average order   :", avg_order)
    print("Top category    :", top_cat)

---
## Well done!

You have used all four control flow tools on real data:

| Tool | Used for |
|------|----------|
| `if / elif / else` | Classifying transactions by revenue size |
| `for` loop | Summarising every store, finding peak hours |
| `while` loop | Tracking cumulative revenue until a target is hit |
| `def` function | Writing reusable, clean, professional code |